# EasyRAG Quickstart

## What you will do

- run a complete EasyRAG lifecycle with deterministic stubs
- inspect grounded citations and retrieval metadata
- see how the same flow changes when you switch to real OpenAI-compatible providers

## Why the notebook starts with stubs

Path A is designed to run without API keys. That keeps the first pass focused on the orchestration shape: initialize, insert, query, inspect, finalize. Path B then shows how to swap in real providers after configuration.

## Step 1: Import the public API

We start with the smallest useful surface area: the public `EasyRAG` API, `QueryParam`, and `run_sync`. The `run_sync` helper matters in notebooks because Jupyter already owns an event loop, while EasyRAG exposes async methods such as `ainsert()` and `aquery()`.


In [ ]:
import json
import tempfile

from easyrag import EasyRAG, QueryParam
from easyrag.support.async_utils import run_sync


You should now have the core teaching tools in scope. There is no meaningful output yet. The important thing is that the notebook is using the same public API that application code would use.

## Path A: Runnable without API keys

The next cells replace real providers with deterministic stub functions. That keeps the example reproducible and makes it easier to see what each stage contributes.


### Step 2: Define a deterministic embedding stub

Dense retrieval normally depends on a real embedding model. For a teaching notebook, that is more infrastructure than we need. This stub converts text into a simple keyword-count vector so retrieval remains stable across runs.


In [ ]:
_KEYWORDS = [
    "architecture",
    "easyrag",
    "embedding",
    "rerank",
    "retrieval",
    "query",
    "rewrite",
    "index",
    "pdf",
    "semantic",
]


def _stub_embedding(texts: list[str]) -> list[list[float]]:
    vectors = []
    for text in texts:
        lowered = text.lower()
        vector = [float(lowered.count(keyword)) for keyword in _KEYWORDS]
        vector.append(float(len(lowered.split())))
        vectors.append(vector)
    return vectors


This cell only defines the function, so it should not print anything. What matters is the shape: one input text becomes one numeric vector. That is enough for EasyRAG's local vector storage to demonstrate dense retrieval behavior.


### Step 3: Define a deterministic query-model stub

EasyRAG can optionally rewrite the incoming question and generate multiple query expansions (MQE). This stub makes both steps visible by returning predictable strings instead of calling a model endpoint.


In [ ]:
def _stub_query_model(prompt: str, *, task: str, count: int = 1) -> str | list[str]:
    cleaned = prompt.split(":", 1)[-1].strip()
    if task == "rewrite":
        return f"{cleaned} repository architecture"
    if task == "mqe":
        return [f"{cleaned} variant {index}" for index in range(1, count + 1)]
    raise ValueError(task)


Again, the result is a definition rather than runtime output. The teaching value is that later metadata will show the rewritten query and the generated variants, so you can see how preprocessing feeds retrieval.


### Step 4: Build a tiny in-memory corpus

We use short in-memory documents instead of scanning the full repository. That keeps the first notebook deterministic and easy to reason about, while still exercising the same `ainsert()` path used by larger workloads.


In [ ]:
demo_texts = [
    "# Architecture\nEasyRAG uses query rewriting and retrieval workflows for repository guidance.\n## Retrieval\nHybrid retrieval uses semantic chunks.\n",
    "# Setup\nUse embeddings and rerank models to answer documentation questions.\n",
    "# Knowledge\nEntity extraction and relation storage can enrich repository search.\n",
]

demo_ids = [
    "doc::architecture",
    "doc::setup",
    "doc::knowledge",
]

demo_paths = [
    "docs/architecture.md",
    "docs/setup.md",
    "docs/knowledge.md",
]

print(json.dumps({"ids": demo_ids, "paths": demo_paths}, indent=2))


When you run the cell, you should see the document IDs and logical file paths. Those metadata fields are useful because EasyRAG carries them through into citations and storage records.


### Step 5: Create and initialize an EasyRAG workspace

We use a temporary working directory so the quickstart stays self-contained. The only special part is dependency injection: `embedding_func` and `query_model_func` are set to the stub functions defined above.


In [ ]:
temp_dir = tempfile.TemporaryDirectory()
working_dir = temp_dir.name
workspace = "quickstart"

rag = EasyRAG(
    working_dir=working_dir,
    workspace=workspace,
    embedding_func=_stub_embedding,
    query_model_func=_stub_query_model,
)

run_sync(rag.initialize_storages())

print(json.dumps({"working_dir": working_dir, "workspace": workspace}, indent=2))


You should see a temporary working directory and the workspace name `quickstart`. At this point, the storages exist but they do not contain any indexed knowledge yet.


### Step 6: Insert the demo documents

`ainsert()` is the main ingestion entry point for raw text. Under the hood, EasyRAG will normalize documents, create chunks, generate summaries, build vector payloads, and produce lightweight entity and relation records.


In [ ]:
insert_stats = run_sync(
    rag.ainsert(
        demo_texts,
        ids=demo_ids,
        file_paths=demo_paths,
    )
)

print(json.dumps(insert_stats, indent=2))


A typical output includes counts such as `documents`, `chunks`, `entities`, and `relations`. The exact numbers are less important than the pattern: one ingestion call fans out into multiple derived retrieval artifacts.


### Step 7: Ask a retrieval question

Now we can run the retrieval side of the pipeline. `QueryParam(mode="hybrid")` tells EasyRAG to combine retrieval signals rather than using only one source. We also keep query rewriting and MQE enabled so their effect becomes visible in the metadata.


In [ ]:
query_text = "How does query rewriting help repository guidance?"
query_param = QueryParam(
    mode="hybrid",
    chunk_top_k=3,
    rewrite_enabled=True,
    mqe_enabled=True,
)

result = run_sync(rag.aquery(query_text, query_param))

print(json.dumps({"mode": result.mode, "citation_count": len(result.citations)}, indent=2))


You should see `mode` set to `hybrid` and at least one citation. That confirms the system is no longer only storing data; it is using that indexed knowledge to answer a question with grounded references.


### Step 8: Inspect grounded citations

Citations are the easiest place to verify that retrieval is working. Each item should carry source metadata such as a title, a location, and a snippet pulled from the inserted corpus.


In [ ]:
print(json.dumps(result.citations[:3], indent=2))


Look for snippets that mention retrieval, query rewriting, or repository guidance. If the cited text tracks the question, the retrieval stage is doing useful work.


### Step 9: Inspect retrieval metadata

The quickstart becomes more instructive when you look at preprocessing metadata. EasyRAG records the normalized query, the rewritten query, the expanded variants, and the retrieval queries that were actually issued.


In [ ]:
interesting_metadata = {
    "original_query": result.metadata.get("original_query"),
    "rewritten_query": result.metadata.get("rewritten_query"),
    "expanded_queries": result.metadata.get("expanded_queries"),
    "retrieval_queries": result.metadata.get("retrieval_queries"),
    "vector_backend": result.metadata.get("vector_backend"),
}

print(json.dumps(interesting_metadata, indent=2))


You should see the rewritten query and several expanded queries derived from the stub model. This is the clearest illustration of how preprocessing changes what the retriever actually searches for.


### Step 10: Finalize and clean up

Even in a tiny notebook, it is worth closing storages explicitly. This mirrors real application lifecycle management and prevents the temporary working directory from hanging around after the walkthrough.


In [ ]:
run_sync(rag.finalize_storages())
temp_dir.cleanup()

print("Closed storages and removed the temporary working directory.")


After this cell, the stub walkthrough is complete. You have seen the full EasyRAG loop once without any provider setup.

## Path B: Switch to real OpenAI-compatible providers

The next two steps keep the same lifecycle, but they remove the stub helpers and rely on environment-backed provider configuration.


### Step 11: Configure provider environment variables

`OPENAI_API_KEY` is the required baseline. `OPENAI_BASE_URL` is optional if you use the default endpoint, and EasyRAG also supports role-specific model and base-URL overrides when you want different providers for query rewriting, embeddings, reranking, or KG extraction.


In [ ]:
env_template = """export OPENAI_API_KEY=\"YOUR_KEY\"
# Optional shared base URL:
# export OPENAI_BASE_URL=\"https://api.openai.com/v1\"

# Optional role-specific base URLs:
# export EASYRAG_QUERY_BASE_URL=\"https://your-query-endpoint/v1\"
# export EASYRAG_EMBEDDING_BASE_URL=\"https://your-embedding-endpoint/v1\"
# export EASYRAG_RERANK_BASE_URL=\"https://your-rerank-endpoint/v1\"
# export EASYRAG_KG_BASE_URL=\"https://your-kg-endpoint/v1\"

# Optional role-specific model names:
# export EASYRAG_QUERY_MODEL_NAME=\"gpt-4.1-mini\"
# export EASYRAG_EMBEDDING_MODEL_NAME=\"text-embedding-3-small\"
# export EASYRAG_RERANK_MODEL_NAME=\"qwen3-rerank\"
# export EASYRAG_KG_MODEL_NAME=\"gpt-4.1-mini\"
"""

print(env_template)


Running this cell should print a copy-pasteable template rather than contacting any endpoint. Fill in the variables in your shell or notebook environment before you execute the real-provider skeleton below.


### Step 12: Keep the same lifecycle, swap the providers

This cell preserves the same order as Path A: create `EasyRAG`, initialize, insert, query, finalize. The only difference is that `EasyRAG()` now uses its default provider stack, which expects the environment variables shown above.


In [ ]:
api_ready = False

if api_ready:
    real_rag = EasyRAG()
    run_sync(real_rag.initialize_storages())
    try:
        run_sync(real_rag.ainsert(demo_texts, ids=demo_ids, file_paths=demo_paths))
        real_result = run_sync(
            real_rag.aquery(
                "How does query rewriting help repository guidance?",
                QueryParam(mode="hybrid"),
            )
        )
        print(json.dumps(real_result.citations[:3], indent=2))
    finally:
        run_sync(real_rag.finalize_storages())
else:
    print("Set OPENAI_API_KEY and any optional overrides first, then change api_ready to True.")


With `api_ready = False`, this cell should only print a reminder. After configuration, changing it to `True` will run the real-provider path with the same ingestion and retrieval lifecycle you already practiced.


## Next steps

- Read [00-overview.md](../../docs/00-overview.md) for the big-picture map of the repository.
- Continue with [02_01_repo_loading.ipynb](../02_data_loading/02_01_repo_loading.ipynb) once you want to understand how source material becomes canonical `Document` objects.
- Keep [notebooks/README.md](../README.md) open as the stage-by-stage notebook roadmap.